In [ ]:
import pandas as pd
import glob
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Find the Excel file automatically (name is long, this avoids typos)
excel_file = glob.glob("../data/raw/tmdb_movies_2024_*_filtered.xlsx")[0]
print(f"Loading: {excel_file}")

df = pd.read_excel(excel_file)

print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")
df.head()

# Data Cleaning
df["budget"] = pd.to_numeric(df["budget"], errors="coerce")
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
df = df[(df["budget"] > 0) & (df["revenue"] > 0)].copy()

# ROI Calculation
df["ROI"] = (df["revenue"] - df["budget"]) / df["budget"]

# Average ROI by Production Country
country_roi = (
    df.assign(prod_country=df["production_countries"].fillna(""").str.split(","))
    .explode("prod_country")
    .assign(prod_country=lambda x: x["prod_country"].str.strip())
    .loc[lambda x: x["prod_country"].ne("")]
    [["prod_country", "ROI"]]
)

avg_roi_by_country = (
    country_roi.groupby("prod_country")["ROI"]
    .mean()
    .dropna()
    .sort_values(ascending=False)
)

top_countries = avg_roi_by_country.head(15)

fig, ax = plt.subplots(figsize=(12, 6))
top_countries.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Average ROI by Production Country")
ax.set_xlabel("Production Country")
ax.set_ylabel("Average ROI")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# Genre Analysis
genre_series = (
    df["genres"]
    .fillna("")
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
    .loc[lambda s: s != ""]
)

# Count each genre and convert to percentages
genre_counts = genre_series.value_counts()
genre_pct = genre_counts / genre_counts.sum() * 100

# Show max and min genres
max_genre = genre_pct.idxmax()
min_genre = genre_pct.idxmin()
max_pct = genre_pct.max()
min_pct = genre_pct.min()

print(f"Highest percentage genre: {max_genre} ({max_pct:.2f}%)")
print(f"Lowest percentage genre: {min_genre} ({min_pct:.2f}%)")

# Plot pie chart
plt.figure(figsize=(10, 8))
plt.pie(
    genre_pct.head(10).values,
    labels=genre_pct.head(10).index,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.6,
)
plt.title("Percentage of Movies by Genre")
plt.axis("equal")
plt.show()

# Regression Analysis
sub = df[['budget', 'revenue', 'belongs_to_collection']].copy().dropna()
sub = sub[(sub['budget'] > 0) & (sub['revenue'] > 0)].copy()

sub['franchise'] = sub['belongs_to_collection'].notna().astype(int)
sub['log_budget'] = np.log1p(sub['budget'])
sub['log_revenue'] = np.log1p(sub['revenue'])

X = sm.add_constant(sub[['log_budget', 'franchise']])
y = sub['log_revenue']
model = sm.OLS(y, X).fit()

print('rows_used =', len(sub))
print('budget_coef =', model.params['log_budget'])
print('franchise_coef =', model.params['franchise'])
print('franchise_p_value =', model.pvalues['franchise'])
print('r_squared =', model.rsquared)

# Approximate multiplicative premium from regression coefficient
prem = (np.exp(model.params['franchise']) - 1) * 100
print('approx_revenue_premium_percent =', round(prem, 4))

model.summary()